In [1]:
import pandas as pd
from backend.graph import graph
from backend.llm import generation_llm
import logging

# Filter out background logs for a clean report
logging.getLogger("langgraph").setLevel(logging.ERROR)

print("✅ Environment ready. Evaluation Harness loaded.")

✅ Environment ready. Evaluation Harness loaded.


In [2]:
TEST_CASES = [
    {"q": "What is my full name?", "expected": "Arindam Das"},
    {"q": "Where do I work currently?", "expected": "Tesla"},
    {"q": "Which company did I work for before Tesla?", "expected": "Google"},
    {"q": "What is my dog's name?", "expected": "Leo"},
    {"q": "What is my favorite comfort food?", "expected": "Rice and Dal"},
    {"q": "Do I prefer tea or coffee in the morning now?", "expected": "Coffee"},
    {"q": "What happened to my plans for Dubai?", "expected": "Cancelled due to new job"},
    {"q": "Where am I planning to travel in 2027?", "expected": "Japan"},
    {"q": "Which university did I get my Masters from?", "expected": "Jadavpur University"},
    {"q": "What is my favorite hobby?", "expected": "Photography"}
]

In [5]:
import time  # <--- Added at the top
from backend.database import get_connection, create_session

results = []
correct_count = 0

# --- NEW: Register the Evaluation Session in the Database ---
connection = get_connection()
try:
    # This creates a real session in your MySQL sessions table
    eval_session_id = create_session(connection)
    print(f"Created evaluation session: {eval_session_id}")
finally:
    connection.close()
# -----------------------------------------------------------

for i, test in enumerate(TEST_CASES):
    print(f"Testing {i+1}/{len(TEST_CASES)}: {test['q']}")
    
    # --- Added to avoid Groq Rate Limits ---
    time.sleep(2) 
    
    # Prepare state using the valid session ID we just created
    state = {
        "user_id": "demo_user",
        "session_id": eval_session_id, # Use the real ID here
        "user_message": test['q'],
        "ai_response": "",
        "conversation_id": 0, "chat_number": 0,
        "episodic_memories": [], "long_term_memories": [],
        "context": "", "conversation_embedding": [],
        "memory_candidate": [], "memory_match": None,
        "memory_decision": None, "memory_decision_list": []
    }

    # Run the Bot
    output = graph.invoke(state)
    actual = output["ai_response"]

    # Use LLM-as-a-Judge to verify
    judge_prompt = f"""
    Compare the AI response to the Ground Truth fact.
    GROUND TRUTH: {test['expected']}
    AI RESPONSE: {actual}
    
    Is the AI correct? Answer with ONLY 'CORRECT' or 'INCORRECT'.
    """
    
    # Using the 120B model as the judge
    verdict_response = generation_llm.invoke(judge_prompt)
    verdict = verdict_response.content.strip().upper()
    
    is_correct = "CORRECT" in verdict
    if is_correct: 
        correct_count += 1
    
    results.append({
        "Question": test['q'],
        "Expected Fact": test['expected'],
        "Bot Response": actual[:150] + "...", 
        "Status": "✅ PASS" if is_correct else "❌ FAIL"
    })

accuracy = (correct_count / len(TEST_CASES)) * 100
print(f"\n✅ Evaluation Complete. Accuracy: {accuracy}%")

Created evaluation session: session_f4c0c284d73b4e90
Testing 1/10: What is my full name?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



[Router] No personal facts to process. Ending.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Testing 2/10: Where do I work currently?


...

Testing 10/10: What is my favorite hobby?


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
INFO:groq._base_client:Retrying request to /openai/v1/chat/completions in 6.000000 seconds
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



[Router] No personal facts to process. Ending.


INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



✅ Evaluation Complete. Accuracy: 100.0%


In [6]:
df = pd.DataFrame(results)
print(f"\n--- AI COMPANION MEMORY RECALL REPORT ---")
print(f"Overall Accuracy: {accuracy}%\n")

# Display the styled dataframe
df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left')])
])


--- AI COMPANION MEMORY RECALL REPORT ---
Overall Accuracy: 100.0%



,Question,Expected Fact,Bot Response,Status
0,What is my full name?,Arindam Das,Your full name is **Arindam Das**....,✅ PASS
1,Where do I work currently?,Tesla,You’re currently working at **Tesla** as a Senior AI Engineer. 🚗✨...,✅ PASS
2,Which company did I work for before Tesla?,Google,You were previously working at **Google** as a software engineer before moving over to Tesla. 🚀😊...,✅ PASS
3,What is my dog's name?,Leo,Your dog’s name is **Leo**. 🐾...,✅ PASS
4,What is my favorite comfort food?,Rice and Dal,I don’t recall you mentioning that yet. I'd love to know more about your favorite comfort food!...,✅ PASS
5,Do I prefer tea or coffee in the morning now?,Coffee,These days you’ve switched things up—you now prefer coffee over tea in the mornings. ☕️😊...,✅ PASS
6,What happened to my plans for Dubai?,Cancelled due to new job,Hey Arindam! 😊 Your Dubai trip didn’t end up happening – you’ve **cancelled the Dubai trip**. If you’d like to chat about what led to the change or s...,✅ PASS
7,Where am I planning to travel in 2027?,Japan,You’ve got a trip to **Japan** lined up for 2027! 🌸✈️...,✅ PASS
8,Which university did I get my Masters from?,Jadavpur University,You earned your master’s degree from **Jadavpur University**. 🎓...,✅ PASS
9,What is my favorite hobby?,Photography,I don’t recall you mentioning a specific favorite hobby yet. I’d love to hear more about which activity you enjoy most!...,✅ PASS
